# 60. `BackgroundCategory` vs `BackgroundSpec`

**Objectives:**

- Build the same background shape twice: once as a low-level `BackgroundCategory`
  fed to `MultiBackgroundNLL` directly, once as a high-level `BackgroundSpec` fed to
  `FitSession`.
- Confirm both give the exact same NLL at the same parameter point.
- See concretely what `FitSession` automates: shape evaluation and normalization on
  the fit's own measure.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV²,
daughter indices start at zero.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede any numerical work: amplitudes use complex128.

import jax.numpy as jnp
import numpy as np

from dalitzplotfitter import (
    BackgroundCategory, BackgroundSpec, DecayChannel, DecayModel, FitSession,
    MultiBackgroundNLL, NonResonant, Parameter, RealImag, Resonance,
    generate_toy,
)
from dalitzplotfitter.integration import GridIntegrator
from dalitzplotfitter.pdf import SignalPDF

## 1. A small signal model and a toy dataset

A single rho plus a non-resonant term, no efficiency or veto -- these are orthogonal
to the point of this notebook.

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
model = DecayModel(
    channel,
    [Resonance("rho", (0, 1), RealImag(1.0, 0.0), mass=0.7753, width=0.1491, spin=1),
     NonResonant(RealImag(0.4, -0.2), name="NR")],
    normalization_method="square-dalitz", normalization_pair=(0, 1),
    normalization_resolution=60,
)
truth = {p.name: p.value for p in model.parameters}
data = generate_toy(
    model, 1500, parameters=truth, seed=60,
    method="inverse-transform", inverse_resolution=256, include_momenta=False,
)
print(f"Generated {data.size} signal-like events")

Generated 1500 signal-like events


## 2. One background shape, defined once

A flat combinatorial shape over the Dalitz plane. `BackgroundSpec` (high-level) takes
this exact callable; `BackgroundCategory` (low-level) instead wants the shape already
evaluated -- on the data sample as `values`, and reduced to a single `normalization`
scalar on whatever sample the fit integrates over.

In [3]:
def background_shape(events):
    return jnp.ones_like(events["s12"])

fraction = Parameter("signal_fraction", 0.75, bounds=(0.05, 0.99), step=0.02)

## 3. High-level: `BackgroundSpec` inside `FitSession`

`FitSession` normalizes `background_shape` for us, automatically, on the fit's own
measure (`model.normalization_sample` here, since no explicit
`normalization_sample=` is given and no veto is applied).

In [4]:
session = FitSession(
    model, data, signal_fraction=fraction,
    backgrounds=(BackgroundSpec("comb", background_shape),),
)
nll_high_level = float(session.objective(truth))
print(f"FitSession + BackgroundSpec: NLL(truth) = {nll_high_level:.6f}")

FitSession + BackgroundSpec: NLL(truth) = 1536.010937


## 4. Low-level: `BackgroundCategory` + `MultiBackgroundNLL` + `SignalPDF`

By hand, this is exactly what `FitSession` did in step 3:

1. Evaluate `background_shape` on the data sample -> `values`.
2. Evaluate it on the model's own normalization sample and reduce with the same
   `integral(f) = mean(weights * f)` convention as everywhere else in this codebase
   (CLAUDE.md, "Normalization: the central invariant") -> `normalization`.
3. Build the signal density as a `SignalPDF` over the same `GridIntegrator`, and feed
   both into `MultiBackgroundNLL` alongside the same `signal_fraction`.

In [5]:
norm_sample = model.normalization_sample
background_values = background_shape(data.as_dict())
background_normalization = jnp.mean(norm_sample.weights * background_shape(norm_sample.as_dict()))

comb = BackgroundCategory(
    "comb", values=background_values, normalization=background_normalization,
)

signal_pdf = SignalPDF(
    intensity=lambda events, values: model.intensity(events, values),
    integrator=GridIntegrator(norm_sample),
)
data_dict = data.as_dict()

def signal_density(values):
    return signal_pdf(data_dict, values)

low_level_nll = MultiBackgroundNLL(
    signal_density=signal_density,
    backgrounds=(comb,),
    signal_fraction=fraction,
)
nll_low_level = float(low_level_nll(truth))
print(f"BackgroundCategory + MultiBackgroundNLL: NLL(truth) = {nll_low_level:.6f}")

np.testing.assert_allclose(nll_low_level, nll_high_level, rtol=1e-10)
print("Both constructions give the identical NLL at the same parameter point.")

BackgroundCategory + MultiBackgroundNLL: NLL(truth) = 1536.010937
Both constructions give the identical NLL at the same parameter point.


## 5. What `FitSession`/`BackgroundSpec` automated

`BackgroundCategory` is a *result*: a fixed `values` array plus a scalar
`normalization`, valid only for the sample it was built against. `BackgroundSpec` is a
*recipe* (the callable shape itself, plus how to normalize and vet it) that
`FitSession` re-evaluates against whatever `data`/`normalization_sample`/veto it is
given -- which is what makes it safe to reuse across `with_constraint(...)` copies or
different datasets without manually re-deriving `values`/`normalization` each time.

## Continue learning

See [`docs/backgrounds_and_vetoes.md`](../../docs/backgrounds_and_vetoes.md) for
multiple background categories, extended-mode yields, and vetoed backgrounds, and
[tutorial 4](tutorial_04_acceptance_and_backgrounds.ipynb) for `BackgroundSpec` combined
with efficiency, a veto and a Gaussian constraint.

Return to [the course guide](TUTORIALS.md).